<a href="https://colab.research.google.com/github/JosianeSousa87/projetos_ad05/blob/main/streaming_de_filmes/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Passo 01. Instalar

#### Aqui vamos instalar a biblioteca que será utilizada:

In [1]:
#Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

#Passo 02. Importações e Autenticação (service account)

#### Fazer a importação e autenticação, autorizando o notebook a acessar a secret GCP_SA_JSONN

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@streamingdefilmes.iam.gserviceaccount.com


#Passo 03. Abrir a planilha

In [3]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1eYe8ygj_o8fqh0ZtTWrNbO_UGPfI1X0uzw_VP5csDww"
NOME_TABELA = "filmes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Streaming_Filmes | filmes | https://docs.google.com/spreadsheets/d/1eYe8ygj_o8fqh0ZtTWrNbO_UGPfI1X0uzw_VP5csDww


#Passo 04. Escrever o cabeçalho

In [5]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["filme_id", "titulo", "genero","ano_lancamento","duracao_min","classificacao_indicativa","diretor_ficticio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


# Passo 05. Gerar e escrever 10 linhas de teste

In [10]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "filme_id": i,
        "titulo": fake.catch_phrase(),
        "genero": random.choice(["ação", "comédia", "drama", "terror", "documentário", "animação", "ficção científica", "romance"]),
        "ano_lancamento": random.randint(1985, 2024), # Garante o ano exatamente entre 1985 e 2024
        "duracao_min": random.randint(75, 184),
        "classificacao_indicativa": random.choice(["Livre", "10", "12", "14", "16", "18"]),
        "diretor_ficticio": fake.name()
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


#Passo 06. Conferir o resultado na planilha

In [11]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 21
Cabeçalho confere: True
[['filme_id', 'titulo', 'genero', 'ano_lancamento', 'duracao_min', 'classificacao_indicativa', 'diretor_ficticio'], ['1', 'A segurança de inovar simplesmente', 'comédia', '2005', '83', '18', 'Srta. Ísis Borges'], ['2', 'A segurança de conseguir mais rapidamente', 'documentário', '2005', '111', '10', 'Amanda Novais']]
